# How many things is this actually measuring?

*Question, Intuition, Math, Code, Assumptions, How it breaks*

The whole project rests on a sentence: **a player must clear a floor on all
eleven requirements**. Eleven independent hurdles, each one a different demand,
and only 6% of players clear every one.

That sentence has a load-bearing word in it, and it is not "eleven". It is
"independent", which I never checked.

## 1. Question

Eleven requirements go into the gate. How many separate things do they actually
test?

## 2. Intuition

Here is the arithmetic that should have bothered me from the beginning.

Each requirement rejects the bottom 40% of players. If the eleven were
independent, the share clearing all of them would be $0.6^{11}$, which is
**0.36%**, or about twenty players out of five and a half thousand. The real
figure is 6.2%, which is seventeen times higher.

That gap is not a mistake. It is correlation, and it is exactly what you would
expect: good players tend to be good at several things at once. But it means the
gate is nothing like eleven hurdles. It is a smaller number of hurdles wearing
eleven labels, and until you know how many, "must have all eleven" is a claim
about the label count rather than about football.

There is a second thing worth finding out. If two requirements measure the same
underlying quality, then the composite score counts that quality twice, and a
player strong in it is rewarded twice for one virtue.

## 3. Math

Three measurements, from weakest to most useful.

**The correlation matrix** $C$, on the standardised career profile the gate
actually sees. Eleven by eleven, one entry per pair.

**Eigenvalues.** $C$ has eleven of them and they sum to eleven. If the
requirements were independent every eigenvalue would be 1. Concentration in the
first few means the variation lives in fewer dimensions than there are columns.
The **participation ratio** turns that into one number:

$$k_{\text{eff}} = \frac{\left(\sum_i \lambda_i\right)^2}{\sum_i \lambda_i^2}$$

**Equivalent independent requirements.** The most directly interpretable of the
three. If the observed pass rate is $p$ and each floor passes 60%, then the gate
behaves like $k$ independent floors where $0.6^k = p$:

$$k = \frac{\ln p}{\ln 0.6}$$

That last one answers the question in the units the claim was made in.

## 4. Code

### The matrix

In [1]:
import numpy as np
import pandas as pd

from gambeta import needs

ranking = pd.read_parquet("../data/sample/ranking.parquet")
keys = [r.key for r in needs.OUTFIELD]
labels = {r.key: r.label for r in needs.OUTFIELD}

C = ranking[keys].corr()
C.rename(index=labels, columns=labels).style.format("{:.2f}").background_gradient(
    cmap="RdBu_r", vmin=-1, vmax=1, axis=None
)

,Scores goals,Creates goals,Finishes clinically,Generates threat,Carries his team,Beats his team's level,Is available,Is picked to start,Sustains it,Has no bad seasons,Does not cost his team
Scores goals,1.00,0.57,0.57,0.94,0.92,1.00,-0.15,-0.43,0.06,0.82,-0.12
Creates goals,0.57,1.00,0.30,0.67,0.52,0.57,-0.12,-0.36,0.12,0.59,-0.06
Finishes clinically,0.57,0.30,1.00,0.46,0.56,0.57,-0.08,-0.21,0.08,0.51,-0.20
Generates threat,0.94,0.67,0.46,1.00,0.90,0.94,-0.18,-0.48,0.05,0.79,-0.16
Carries his team,0.92,0.52,0.56,0.90,1.00,0.92,-0.00,-0.31,0.06,0.81,-0.12
Beats his team's level,1.00,0.57,0.57,0.94,0.92,1.00,-0.15,-0.43,0.06,0.82,-0.12
Is available,-0.15,-0.12,-0.08,-0.18,-0.00,-0.15,1.00,0.72,0.28,0.23,0.30
Is picked to start,-0.43,-0.36,-0.21,-0.48,-0.31,-0.43,0.72,1.00,0.20,-0.06,0.27
Sustains it,0.06,0.12,0.08,0.05,0.06,0.06,0.28,0.20,1.00,0.11,0.06
Has no bad seasons,0.82,0.59,0.51,0.79,0.81,0.82,0.23,-0.06,0.11,1.00,0.18


Read the top-left block first. `scoring`, `threat`, `team_share`, `above_team`
and `consistency` sit between 0.79 and 1.00 with each other. That is not eleven
requirements, it is one attacking quality measured five ways.

Then the pair in the middle: `availability` and `reliability` at **0.72**. Being
on the pitch and being picked to start are, as measured here, largely the same
fact about a player.

And one entry that should not be possible.

In [2]:
# Upper triangle only, so each pair is named once.
upper = C.where(~np.tril(np.ones(C.shape, dtype=bool)))
pairs = (
    upper.melt(ignore_index=False, var_name="other", value_name="r")
    .dropna()
    .set_index("other", append=True)["r"]
    .sort_values(key=abs, ascending=False)
)
print("the six strongest relationships between supposedly separate requirements:\n")
for (a, b), v in pairs.head(6).items():
    print(f"  {labels[a]:<24} {labels[b]:<24} {v:+.4f}")

the six strongest relationships between supposedly separate requirements:

  Scores goals             Beats his team's level   +0.9999
  Scores goals             Generates threat         +0.9437
  Generates threat         Beats his team's level   +0.9437
  Scores goals             Carries his team         +0.9185
  Carries his team         Beats his team's level   +0.9184
  Generates threat         Carries his team         +0.9021


**`scoring` and `above_team` correlate at 0.9999.**

Requirement 1 is "scores goals". Requirement 6 is "beats his team's level", and
it is built as the residual of scoring after regressing on the club's ClubElo
rating. The intention was to separate the player from the side around him: score
20 for a title winner and it should count for less than 20 for a relegation
side.

It does not do that, because within a single league-season, club strength
explains almost none of the variation in who scores. Take the residual of
something that was barely explained and you get the thing back. Then both get
z-scored inside the same group, and the last difference disappears.

So the definition has ten requirements in it, not eleven, and one of them is
counted twice in every composite score.

### How many hurdles is the gate really?

In [3]:
eigenvalues = np.linalg.eigvalsh(C.to_numpy())[::-1]
observed = ranking["qualified"].mean()

print("eigenvalues (they sum to 11; all ones would mean independence):")
print("  " + "  ".join(f"{v:.2f}" for v in eigenvalues))
print()
print(f"participation ratio          {eigenvalues.sum() ** 2 / (eigenvalues**2).sum():.2f}")
print(f"components for 90% variance  {int(np.searchsorted(np.cumsum(eigenvalues) / 11, 0.90) + 1)}")
print()
print(f"pass rate if independent     {0.6**11:.4%}")
print(f"pass rate observed           {observed:.4%}")
print(f"equivalent independent gates {np.log(observed) / np.log(0.6):.2f}")

eigenvalues (they sum to 11; all ones would mean independence):
  5.55  2.04  0.99  0.86  0.62  0.51  0.22  0.10  0.07  0.04  0.00

participation ratio          3.24
components for 90% variance  5

pass rate if independent     0.3628%
pass rate observed           6.2092%
equivalent independent gates 5.44


**Eleven requirements behave like about five and a half.**

The smallest eigenvalue is zero, which is the matrix saying out loud that one
column is a copy of another. That is `above_team`, found independently by the
arithmetic rather than by anybody reading the code.

The three measures disagree with each other, and the disagreement is honest
rather than a problem. The participation ratio weights the largest eigenvalue
heavily and gives about three. Ninety per cent of the variance needs five
components. The pass-rate calculation gives 5.4. They are asking slightly
different questions, and all three land between three and six, which is the
answer: **the gate tests roughly half as many things as it says it does.**

### Does correlation actually explain the gap?

The claim above is that correlation, and nothing else, accounts for 6.2% rather
than 0.36%. That is testable. Draw fake players with exactly this correlation
structure and gate them the same way.

In [4]:
rng = np.random.default_rng(20260810)
draws = rng.multivariate_normal(np.zeros(len(keys)), C.to_numpy(), size=200_000)

floors = np.percentile(draws, 40, axis=0)
simulated = (draws >= floors).all(axis=1).mean()

print(f"independent requirements     {0.6**11:.4%}")
print(f"simulated, this correlation  {simulated:.4%}")
print(f"observed in the real data    {observed:.4%}")

independent requirements     0.3628%
simulated, this correlation  5.4635%
observed in the real data    6.2092%


Close enough to call it. The correlation structure alone reproduces the
qualifier rate, so nothing else needs explaining: the gate is not unexpectedly
generous, it is exactly as generous as eleven overlapping tests should be.

What is left over is worth noticing too. The simulation assumes joint normality
and the real requirements are skewed, which is why the two do not match exactly.

## 5. Assumptions

1. **Correlation captures the dependence.** Pearson sees linear relationships.
   Two requirements could be strongly dependent in a curved way and show a
   correlation near zero.
2. **The career profile is the right place to measure.** These are pooled,
   standardised career values, which is what the gate sees. The season-level
   picture is similar but not identical.
3. **The simulation's normality is wrong and useful anyway.** It is there to show
   that correlation is sufficient to explain the pass rate, not to model the
   distribution faithfully.

## 6. How it breaks

**Effective dimensionality is not a verdict on any single requirement.**

It would be easy to read "five and a half" as "delete five requirements", and
that does not follow. A requirement can be highly correlated with others and
still be the one doing the eliminating at the margin. `discipline` correlates
weakly with everything, and it rejects more players on its own than any other.
Correlation describes the population; the gate acts on individuals.

The one case where the reading is safe is a correlation of 0.9999, because there
is no margin left for a duplicate to act differently at.

**What this chapter changes, and what it leaves open.**

It settles that `above_team` is not a separate requirement, which is a fact
rather than a judgement. What to do about it is a judgement: drop it and rank on
ten, or rebuild it against something club strength actually predicts, which
would mean a different measure of a team's level than a rating that barely moves
within a season.

It also hands the open question about `reliability` a second piece of evidence.
The sensitivity chapter found that `reliability` is the requirement that gives
way first whenever any threshold is tightened. This one finds it correlates 0.72
with `availability`. A requirement that is both fragile and largely redundant is
a requirement with a case to answer.

Neither is decided here. Both are now measured, which is the difference between
an argument and an opinion.